# 从边缘检测到车道分割

学习顺序：梯度算子 → 二阶导数 → Canny → ROI/Hough → CNN → SCNN。
先运行全部单元，再只改一个参数做对比。现有输入是 `img.jpg`；缺失时明确报错。
固定卷积核只提取局部变化，边缘既可能来自车道，也可能来自车、树、阴影。
深度学习用标注学习“哪些像素是车道”，而不是简单把所有边缘当作车道。


In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

image_path = Path("img.jpg")
bgr = cv2.imread(str(image_path))
if bgr is None:
    raise FileNotFoundError(f"无法读取 {image_path.resolve()}，请在项目目录运行 Notebook")
img = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

def show_grid(images, columns=3, figsize=(15, 8)):
    rows = (len(images) + columns - 1)//columns
    fig, axes = plt.subplots(rows, columns, figsize=figsize, squeeze=False)
    for ax, (title, image) in zip(axes.flat, images.items()):
        ax.imshow(image, cmap="gray" if image.ndim == 2 else None)
        ax.set_title(title)
        ax.axis("off")
    for ax in list(axes.flat)[len(images):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_grid({"RGB input": cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), "Gray": img}, columns=2)


# Gradient based edge detection

In [ ]:


# ============================================================
# 1. Roberts 算子（2×2 核，对角线差分）
# Gx = [[1, 0], [0, -1]] * f
# Gy = [[0, 1], [-1, 0]] * f
# G = sqrt(Gx² + Gy²)
# ============================================================
kernel_roberts_x = np.array([[1, 0],
                              [0, -1]], dtype=np.float32)
kernel_roberts_y = np.array([[0, 1],
                              [-1, 0]], dtype=np.float32)

roberts_x = cv2.filter2D(img, cv2.CV_64F, kernel_roberts_x)
roberts_y = cv2.filter2D(img, cv2.CV_64F, kernel_roberts_y)
edge_roberts = np.sqrt(roberts_x**2 + roberts_y**2)
edge_roberts = np.clip(edge_roberts, 0, 255).astype(np.uint8)

# ============================================================
# 2. Prewitt 算子（3×3 核，无中心加权）
# Gx = [[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]] * f
# Gy = [[-1, -1, -1], [0, 0, 0], [1, 1, 1]] * f
# G = sqrt(Gx² + Gy²)
# ============================================================
kernel_prewitt_x = np.array([[-1, 0, 1],
                              [-1, 0, 1],
                              [-1, 0, 1]], dtype=np.float32)
kernel_prewitt_y = np.array([[-1, -1, -1],
                              [ 0,  0,  0],
                              [ 1,  1,  1]], dtype=np.float32)

prewitt_x = cv2.filter2D(img, cv2.CV_64F, kernel_prewitt_x)
prewitt_y = cv2.filter2D(img, cv2.CV_64F, kernel_prewitt_y)
edge_prewitt = np.sqrt(prewitt_x**2 + prewitt_y**2)
edge_prewitt = np.clip(edge_prewitt, 0, 255).astype(np.uint8)

# ============================================================
# 3. Sobel 算子（3×3 核，有中心加权，抗噪更好）
# Gx = [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]] * f
# Gy = [[-1, -2, -1], [0, 0, 0], [1, 2, 1]] * f
# G = sqrt(Gx² + Gy²)
# ============================================================
sobel_x = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
edge_sobel = np.sqrt(sobel_x**2 + sobel_y**2)
edge_sobel = np.clip(edge_sobel, 0, 255).astype(np.uint8)

# ============================================================
# 显示
# ============================================================
show_grid({"Gray": img, "Roberts (clipped)": edge_roberts,
           "Prewitt (clipped)": edge_prewitt, "Sobel (clipped)": edge_sobel})


## Scharr 与梯度幅值

Roberts 使用对角差分，Prewitt 使用均匀差分，Sobel 在差分的垂直方向加入平滑权重。
Scharr 的 3×3 权重提高旋转方向上的梯度近似精度，但仍会响应噪声。
求导必须使用有符号浮点数：直接用 uint8 会丢失负梯度。
上面的 clip 适合显示，但会饱和；下面单独归一化只比较形状，不能比较绝对强度。


In [ ]:
scharr_x = cv2.Scharr(img, cv2.CV_64F, 1, 0)
scharr_y = cv2.Scharr(img, cv2.CV_64F, 0, 1)
edge_scharr = cv2.magnitude(scharr_x, scharr_y)

def display_magnitude(values):
    return cv2.normalize(np.abs(values), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

show_grid({"Sobel normalized": display_magnitude(np.hypot(sobel_x, sobel_y)),
           "Scharr normalized": display_magnitude(edge_scharr),
           "Scharr dx": display_magnitude(scharr_x)})


## Laplacian 与 LoG：二阶导数

Laplacian 是二阶导数，正负响应可用于寻找零交叉，通常比一阶导数更敏感于噪声。
LoG（Laplacian of Gaussian）先平滑再求二阶导数。
以下显示绝对响应，不是完整的零交叉边缘检测；负响应不能提前截断。


In [ ]:
laplacian = cv2.Laplacian(img, cv2.CV_64F, ksize=3)
smoothed = cv2.GaussianBlur(img, (5, 5), 1.2)
log_response = cv2.Laplacian(smoothed, cv2.CV_64F, ksize=3)
show_grid({"Gray": img, "abs Laplacian": display_magnitude(laplacian),
           "abs LoG": display_magnitude(log_response)})


## Canny：从梯度到细边缘

流程：高斯去噪 → 梯度幅值/方向 → 非极大值抑制 → 双阈值 → 滞后连接。
超过高阈值的强边缘保留；低高阈值之间的弱边缘只有与强边缘连通才保留。
`L2gradient=True` 使用欧氏梯度幅值，和默认 L1 幅值的阈值含义不同。
阈值低会保留更多噪声，阈值高会丢失弱车道线；没有适用于所有道路的固定阈值。


In [ ]:
blurred = cv2.GaussianBlur(img, (5, 5), 0)
canny_results = {f"Canny {low}/{high}": cv2.Canny(blurred, low, high, L2gradient=True)
                 for low, high in [(30, 90), (50, 150), (100, 200)]}
show_grid(canny_results)


## 噪声实验：保持同一幅值尺度

给灰度图加入可重复的高斯噪声，对比“直接求梯度”和“先去噪再求梯度”。
观察天空和路面的响应：更强的平滑会减少噪声，也会模糊细车道线。


In [ ]:
rng = np.random.default_rng(42)
noisy = np.clip(img.astype(np.float32)+rng.normal(0, 15, img.shape), 0, 255).astype(np.uint8)
denoised = cv2.GaussianBlur(noisy, (5, 5), 1.2)
noisy_gradient = np.hypot(cv2.Sobel(noisy, cv2.CV_32F, 1, 0), cv2.Sobel(noisy, cv2.CV_32F, 0, 1))
clean_gradient = np.hypot(cv2.Sobel(denoised, cv2.CV_32F, 1, 0), cv2.Sobel(denoised, cv2.CV_32F, 0, 1))
show_grid({"Noisy": noisy, "Denoised": denoised,
           "Noisy Sobel clipped": np.clip(noisy_gradient, 0, 255).astype(np.uint8),
           "Denoised Sobel clipped": np.clip(clean_gradient, 0, 255).astype(np.uint8)}, columns=2)


## 把边缘变成车道：ROI、颜色和 Hough

传统方法把“道路在下方、车道为白/黄色、左右线有特定倾斜方向”写成规则。
Hough 把边缘点投票为线段，再用长度加权最小二乘拟合左右两条线。
拟合使用 x=a·y+b；图像坐标 y 向下增大，因此左边界通常 dx/dy<0。
无检测时输出原图，不虚构车道。闭运算可连接间隙，也可能连接无关边缘。


In [ ]:
from Traditional_Lane_Detection import detect_lanes

overlay, steps = detect_lanes(bgr)
show_grid({"Color mask": steps["colors"], "ROI": steps["roi"],
           "Selected edges": steps["selected_edges"],
           "Hough segments": cv2.cvtColor(steps["hough_segments"], cv2.COLOR_BGR2RGB),
           "Fitted lanes": cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)})
print("Fitted endpoints:", steps["lanes"])


In [ ]:
without_color, _ = detect_lanes(bgr, use_color=False)
with_closing, _ = detect_lanes(bgr, close_gaps=True)
show_grid({"Default": cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB),
           "No color filter": cv2.cvtColor(without_color, cv2.COLOR_BGR2RGB),
           "Closing enabled": cv2.cvtColor(with_closing, cv2.COLOR_BGR2RGB)})


## 从手工规则迁移到 CNN / SCNN

| 传统流程 | 深度学习对应 | 学习重点 |
|---|---|---|
| Roberts/Sobel 固定卷积核 | CNN 可学习卷积核 | 卷积、通道、感受野、反向传播 |
| 灰度、颜色阈值 | RGB 输入与学习到的特征 | BGR/RGB 和训练/推理预处理一致 |
| ROI/斜率筛选 | 人工像素标注监督 | 模型从样本中学习，仍可能学到数据偏差 |
| Hough 和直线连接 | SCNN 沿行列传递信息 | 前一切片更新后再传给下一切片 |
| 左右直线端点 | 每个像素的车道概率 | 二分类分割不自动区分车道实例 |

这里的“迁移”首先是方法迁移。教学模型从随机初始化开始；如果要做预训练迁移学习，
后续可换为 ImageNet 预训练骨干，再微调。当前代码不包含预训练权重。
SCNN 思想参考[原论文](https://arxiv.org/abs/1712.06080)与[官方实现](https://github.com/XingangPan/SCNN)。
本项目是小型教学实现，省略原论文骨干、车道存在分支和实例评测。

### 实践顺序

1. 改 Canny 阈值、ROI 和颜色开关，记录在阴影/弯道下失效的原因。
2. 阅读 `deeplearning_lane_detection_SCNN.py`，跟踪 N×3×H×W → N×64×H/8×W/8 → N×1×H×W。
3. 在含 torch 的环境执行 `python deeplearning_lane_detection_SCNN.py smoke-test`，理解 logits、BCE、Dice 和梯度。
4. 准备人工像素标签：同名 `.png`，单通道 0=背景，1 或 255=车道。这里 255 是前景，不是 ignore。
5. 按视频/道路划分 train 与 val；先让模型拟合少量训练样本确认管线，再看独立验证集。
6. 相同数据划分和 seed，比较 `--no-scnn` 的普通 CNN 与 SCNN；分别设置输出权重路径。
7. 比较前景 IoU 和可视化；背景占比高时，像素准确率容易误导。此处 IoU 不能替代 CULane 官方车道实例指标。

```text
data/
  train/images/0001.jpg
  train/masks/0001.png
  val/images/1001.jpg
  val/masks/1001.png
```

```bash
python Traditional_Lane_Detection.py --input img.jpg --debug-dir outputs/steps
python Traditional_Lane_Detection.py --input video.mp4 --output outputs/traditional.mp4
python deeplearning_lane_detection_SCNN.py train --train-dir data/train --val-dir data/val
python deeplearning_lane_detection_SCNN.py train --train-dir data/train --val-dir data/val --no-scnn --output outputs/cnn_best.pt
python deeplearning_lane_detection_SCNN.py predict --checkpoint outputs/scnn_best.pt --input img.jpg
```

依赖：传统/Notebook 使用 numpy、opencv-python、matplotlib；深度学习另需 PyTorch。
当前仅有输入图片和视频，没有人工标签或已训练权重，不能据此声称深度模型有效。

本机环境检查：`CV` 可运行传统算法和本 Notebook；`pytorchEnv` 有 PyTorch，
但当前缺少可用的 OpenCV。请激活 `pytorchEnv` 后执行 `python -m pip install opencv-python`，
确保同一解释器能同时 `import cv2` 与 `import torch`，再运行深度学习脚本。
`outputs/verification_synthetic` 中的权重仅用于合成数据管线验证，不用于真实道路预测。
